# STAC-MJX

We will fit a rodent body model to recorded 3D keypoints. First, we will calibrate
where the model's markers lie on the body. Then we will estimate a pose for each
frame and compute velocities. We will render the fitted motion, measure its
agreement with the recording, and inspect individual joints.

Calibration estimates marker offsets that stay fixed across the recording.
Inverse kinematics (IK) estimates poses that change from frame to frame. We will
run these stages separately so we can inspect and reuse the calibration. Model
scale stays fixed during both stages.

## 1. Prepare the workspace

Use Python 3.12 and install STAC from the repository root:

```bash
uv venv --python 3.12
source .venv/bin/activate
uv pip install -e ".[cuda13,notebooks]"
python -m ipykernel install --user --name stac-mjx --display-name "Python (stac-mjx)"
```

Use `cuda12` for a compatible CUDA 12 installation, or `.[notebooks]` for CPU.
Select that kernel and open this notebook from the repository root or `notebooks/`.

The first fit compiles JAX programs and can take several minutes. A GPU is
recommended for this rodent model; fewer input frames do not remove the initial
compilation cost. Rendering needs an OpenGL backend as described in the
[README](../README.md). The next cell supplies defaults for NVIDIA EGL on Linux
and a desktop window system elsewhere. Existing environment settings take
precedence. Restart the kernel if you change them after importing JAX or MuJoCo.


In [1]:
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("MUJOCO_GL", "egl" if sys.platform == "linux" else "glfw")
if os.environ["MUJOCO_GL"] in {"egl", "osmesa"}:
    os.environ.setdefault("PYOPENGL_PLATFORM", os.environ["MUJOCO_GL"])

In [ ]:
import copy
import importlib.metadata as metadata
import time
from contextlib import redirect_stdout
from pathlib import Path

import h5py
import imageio_ffmpeg
import jax
import matplotlib.pyplot as plt
import mediapy as media
import mujoco
import numpy as np
import pandas as pd
from IPython.display import display
from omegaconf import OmegaConf

import stac_mjx
from stac_mjx.io import load_stac_data

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "stac_mjx" / "version.py").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the stac-mjx repository.")

media.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())
plt.rcParams.update(
    {"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False}
)
display(
    pd.Series(
        {
            name: metadata.version(name)
            for name in ("stac-mjx", "jax", "jaxlib", "mujoco", "mujoco-mjx")
        },
        name="installed version",
    ).to_frame()
)
print("Python:", sys.version.split()[0])
print("JAX backend:", jax.default_backend())
print("Devices:", [device.device_kind for device in jax.devices()])
print("Renderer:", os.environ["MUJOCO_GL"])
print("STAC source:", Path(stac_mjx.__file__).resolve())

### Choose a recording window

The default window contains five seconds of samples at 30 Hz. Calibration uses
its first 50 frames; IK fits all 150. For a new recording, choose calibration
frames that contain useful variation in pose. This short window illustrates the
workflow.

We keep the rodent preset's six calibration rounds and the solver's 150-iteration
limits. Smaller calibration and chunk sizes limit memory use. The settings table
below shows these choices. Set `N_FRAMES = None` to fit all remaining frames.


In [3]:
START_FRAME = 0
N_FRAMES = 150
CALIBRATION_FRAMES = 50
FRAMES_PER_CHUNK = 50
CONTEXT_FRAMES = 8
CAMERA = "close_profile"

output_dir = repo_root / "outputs" / "rodent_tour"
output_dir.mkdir(parents=True, exist_ok=True)

cfg = stac_mjx.load_stac_config(
    repo_root / "configs", overrides=["stac=demo", "model=rodent"]
)
cfg.stac.calibration_path = str(output_dir / "calibration.h5")
cfg.stac.ik_path = str(output_dir / "ik.h5")
cfg.stac.n_calibration_frames = CALIBRATION_FRAMES
cfg.stac.n_frames_per_clip = FRAMES_PER_CHUNK
cfg.stac.q_opt.context_frames = CONTEXT_FRAMES
cfg.stac.skip_calibration = False
cfg.stac.skip_ik = False
cfg.stac.infer_qvels = True
cfg.model.N_SAMPLE_FRAMES = CALIBRATION_FRAMES

recording_path = repo_root / cfg.stac.data_path
model_path = repo_root / cfg.model.MJCF_PATH
print("Recording:", recording_path.name)
print("Model:", model_path.name)
print("Outputs:", output_dir)

Config loaded and validated.
Recording: test_rodent_mocap_1000_frames.nwb
Model: rodent.xml
Outputs: /home/vyom/projects/stac-mjx/outputs/rodent_tour


## 2. Inspect the recording and model

A keypoint is an observed position in world coordinates. A model marker is a
point attached to a body, such as the skull or forearm. The loader matches
configured keypoint names to `KEYPOINT_MODEL_PAIRS`, converts coordinates with
`MOCAP_SCALE_FACTOR`, and returns an array shaped
`(frames, 3 * keypoints)`. Each keypoint contributes adjacent x, y, z columns.

The bundled HDF5 recording stores `tracks` as `(1000, 1, 23, 3)`:
frames, individuals, keypoints, and XYZ coordinates. It also includes `kp_names`,
`timestamps`, and `confidence`. The existing loader takes names from the model
config, so we check that they match the file's stored order before loading.

`configs/stac/demo.yaml` sets `data_fps: 30`. We check that rate against the
HDF5 timestamps and its `data_fps` attribute. Taking consecutive frames preserves
the sample interval used for velocity estimation.

Coordinates are stored in millimetres; the rodent preset converts them to metres
with `MOCAP_SCALE_FACTOR: 0.001`. The HDF5 preserves the original NWB observations
and records the unit interpretation and timing provenance in its attributes.
Confidence scores are retained but are not used as fit weights.



In [ ]:
with h5py.File(recording_path, "r") as recording:
    source_names = recording["kp_names"].asstr()[:].tolist()
    timestamps = np.asarray(recording["timestamps"][:], dtype=float)
    source_units = recording["tracks"].attrs["unit"]
    assert source_names == list(cfg.model.KP_NAMES)
    assert recording["tracks"].shape == (len(timestamps), 1, len(source_names), 3)
    assert source_units == "millimeters"
    np.testing.assert_allclose(cfg.stac.data_fps, recording.attrs["data_fps"])

intervals = np.diff(timestamps)
assert np.isfinite(timestamps).all() and np.all(intervals > 0)
dt = float(np.median(intervals))
np.testing.assert_allclose(intervals, dt, rtol=1e-6, atol=1e-9)
np.testing.assert_allclose(cfg.stac.data_fps, 1 / dt, rtol=1e-6)
cfg.model.RENDER_FPS = round(cfg.stac.data_fps)

all_keypoints, kp_names = stac_mjx.load_keypoint_data(cfg, repo_root)
assert kp_names == list(cfg.model.KEYPOINT_MODEL_PAIRS)
assert all_keypoints.shape == (len(timestamps), 3 * len(kp_names))
stop_frame = len(timestamps) if N_FRAMES is None else START_FRAME + N_FRAMES
assert 0 <= START_FRAME < stop_frame <= len(timestamps)
kp_data = all_keypoints[START_FRAME:stop_frame]
observed = np.asarray(kp_data).reshape(len(kp_data), len(kp_names), 3)
assert np.isfinite(observed).all(), (
    "Resolve missing or nonfinite keypoints before fitting."
)
assert 2 <= CALIBRATION_FRAMES <= len(kp_data)

frame_time = timestamps[START_FRAME:stop_frame]
elapsed = frame_time - frame_time[0]
frame_table = pd.DataFrame(
    {
        "source_frame": np.arange(START_FRAME, stop_frame),
        "recording_time_s": frame_time,
        "elapsed_time_s": elapsed,
    }
)
frame_table.to_csv(output_dir / "frames.csv", index=False)
display(
    pd.Series(
        {
            "recorded frames": len(timestamps),
            "selected frames": len(kp_data),
            "keypoints": len(kp_names),
            "sampling frequency (Hz)": cfg.stac.data_fps,
            "HDF5 coordinate units": source_units,
            "coordinate multiplier": cfg.model.MOCAP_SCALE_FACTOR,
            "model scale": cfg.model.SCALE_FACTOR,
        },
        name="value",
    ).to_frame()
)

In [ ]:
marker_colors = [
    np.fromstring(cfg.model.KEYPOINT_COLOR_PAIRS[name], sep=" ") for name in kp_names
]
selected = np.linspace(0, len(observed) - 1, 3, dtype=int)
points_mm = observed * 1000
center = (points_mm.min(axis=(0, 1)) + points_mm.max(axis=(0, 1))) / 2
half_width = max(np.ptp(points_mm, axis=(0, 1)).max() / 2, 1.0)

fig = plt.figure(figsize=(13, 5))
fig.subplots_adjust(left=0.02, right=0.96, bottom=0.15, top=0.87, wspace=0.2)
for panel, frame in enumerate(selected, start=1):
    ax = fig.add_subplot(1, 3, panel, projection="3d")
    ax.scatter(*points_mm[frame].T, c=marker_colors, s=18)
    snout = points_mm[frame, kp_names.index("Snout")]
    ax.text(*snout, "Snout", fontsize=8)
    ax.set(
        xlim=(center[0] - half_width, center[0] + half_width),
        ylim=(center[1] - half_width, center[1] + half_width),
        zlim=(center[2] - half_width, center[2] + half_width),
        xlabel="x (mm)",
        ylabel="y (mm)",
        zlabel="z (mm)",
        title=f"t = {elapsed[frame]:.2f} s",
    )
    ax.set_box_aspect((1, 1, 1))
fig.suptitle("Observed keypoints in world coordinates")
fig.savefig(output_dir / "observed_keypoints.png", dpi=150)
plt.show()

### Map keypoints to bodies

The table links each observation to a body in the MuJoCo model. Its marker offset
is expressed in that body's local coordinates, in meters. Several markers can
attach to the same body.

`SCALE_FACTOR` changes model size; `MOCAP_SCALE_FACTOR` converts input
coordinates. These have different purposes. Calibration adjusts marker offsets,
not bone lengths or model scale. Root initialization uses `SpineL` for placement
and the configured trunk keypoints for orientation.


In [ ]:
model = mujoco.MjModel.from_xml_path(str(model_path))
initial_offsets = np.array(
    [
        np.fromstring(cfg.model.KEYPOINT_INITIAL_OFFSETS[name], sep=" ")
        for name in kp_names
    ]
)
mapping = pd.DataFrame(
    {
        "keypoint": kp_names,
        "body": [cfg.model.KEYPOINT_MODEL_PAIRS[name] for name in kp_names],
        "regularized offset": [
            name in cfg.model.SITES_TO_REGULARIZE for name in kp_names
        ],
    }
)
display(mapping)
print(
    f"Model: {model.njnt} joints, {model.nq} position coordinates, {model.nv} velocities"
)
print("Trunk keypoints:", ", ".join(cfg.model.TRUNK_OPTIMIZATION_KEYPOINTS))
display(
    pd.Series(
        {
            "calibration frames": cfg.stac.n_calibration_frames,
            "calibration rounds": cfg.model.N_ITERS,
            "frames per offset update": cfg.model.N_SAMPLE_FRAMES,
            "IK frames per chunk": cfg.stac.n_frames_per_clip,
            "context frames on each side": cfg.stac.q_opt.context_frames,
            "coarse initialization stride": cfg.stac.q_opt.coarse_init_stride,
            "calibration iteration limit": cfg.stac.q_opt.calibration_max_iterations,
            "IK iteration limit": cfg.stac.q_opt.ik_max_iterations,
            "temporal residual weight": cfg.stac.q_opt.velocity_smoothness_weight,
        },
        name="value",
    ).to_frame()
)

## 3. Calibrate the marker offsets

Let $q_t$ be the pose at frame $t$, and $m_k$ the local offset of marker $k$.
If that marker belongs to body $b$, its predicted world position is

$$\hat y_{tk} = p_b(q_t) + R_b(q_t)m_k.$$

Here $p_b$ is the body's position and $R_b$ its rotation. STAC fits these
predictions to the observed positions $y_{tk}$.

Calibration starts by placing the root. It then alternates two steps: fit poses
with offsets fixed, and fit offsets with poses fixed. A final pose fit uses the
updated offsets. `N_ITERS` sets the number of alternating rounds;
`N_SAMPLE_FRAMES` limits the frames used for each offset update.

`SITES_TO_REGULARIZE` selects offsets whose changes receive a penalty.
`M_REG_COEF` controls that penalty. In the current implementation, each offset
update uses the preceding round's offsets as its reference.

We set `skip_ik=True` to stop after calibration. The HDF5 file stores the
calibration poses, offsets, keypoints, and configuration. Detailed solver output
goes to `calibration.log`.


In [ ]:
calibration_cfg = copy.deepcopy(cfg)
calibration_cfg.stac.skip_ik = True
started = time.perf_counter()
with (output_dir / "calibration.log").open("w") as log, redirect_stdout(log):
    calibration_path, skipped_ik_path = stac_mjx.run_stac(
        calibration_cfg, kp_data, kp_names, base_path=repo_root
    )
calibration_seconds = time.perf_counter() - started
assert skipped_ik_path is None
_, calibration = load_stac_data(calibration_path)
assert calibration.qpos.shape == (CALIBRATION_FRAMES, model.nq)
assert calibration.kp_names == kp_names
assert np.isfinite(calibration.qpos).all()
assert np.isfinite(calibration.offsets).all()
print(f"Calibration: {calibration_seconds:.1f} s, including any compilation")
print("Saved:", Path(calibration_path).name)
print("Solver log:", output_dir / "calibration.log")

### Inspect the offset changes

The plot measures how far each marker moved in its body's local coordinates.
Several initial offsets are zero, placing those markers at their body origins.
Moving them toward observed landmarks can require a large change. Large changes
can also indicate a poor initial offset, a scale mismatch, or a marker
absorbing errors elsewhere in the model. The change alone is not a measure of
calibration accuracy. We will inspect the fitted motion and marker errors next.


In [ ]:
offset_change_mm = (np.asarray(calibration.offsets) - initial_offsets) * 1000
offset_table = mapping.copy()
offset_table[["dx (mm)", "dy (mm)", "dz (mm)"]] = offset_change_mm
offset_table["change (mm)"] = np.linalg.norm(offset_change_mm, axis=1)
offset_table.to_csv(output_dir / "offset_changes.csv", index=False)
display(offset_table.round(3))

order = np.argsort(offset_table["change (mm)"].to_numpy())
fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
ax.barh(
    np.array(kp_names)[order],
    offset_table["change (mm)"].to_numpy()[order],
    color=[
        "#dc8b24" if offset_table["regularized offset"][i] else "#137c8b" for i in order
    ],
)
ax.set(
    xlabel="Offset change (mm)",
    title="Marker calibration · orange marks regularized offsets",
)
fig.savefig(output_dir / "offset_changes.png", dpi=150)
plt.show()

## 4. Fit the motion

We now hold the calibrated offsets fixed and fit every selected frame.
`skip_calibration=True` loads the saved offsets. It also checks that their
keypoint names match the current ordering. For reuse, the body model, scale, and
marker definitions must remain compatible.

IK solves chunks with context on either side, then keeps each chunk's central
frames. Adjacent chunks therefore share observations near their boundaries.
Coarse initialization first fits sparse frames and interpolates their poses to
seed the full solve. Interpolation respects rotational joints.

`velocity_smoothness_weight` weights differences between adjacent poses.
A larger value discourages rapid changes but can increase marker error. This
term uses pose differences, not measured velocities divided by the recording
timestep. Its effect therefore depends on sampling frequency.

### Compute velocities at the recording rate

With `infer_qvels=True`, STAC differentiates the fitted poses using
$\Delta t = 1 / \mathrm{data\_fps}$. For translation and scalar joints, the
estimate is a forward difference. Rotational joints use relative rotations
instead of subtracting quaternion components. The final frame uses the last
backward difference.

The MuJoCo simulation timestep does not set this interval. Keep
`data_fps` equal to the rate of the frames supplied to STAC.


In [ ]:
ik_cfg = copy.deepcopy(cfg)
ik_cfg.stac.skip_calibration = True
started = time.perf_counter()
with (output_dir / "ik.log").open("w") as log, redirect_stdout(log):
    _, ik_path = stac_mjx.run_stac(ik_cfg, kp_data, kp_names, base_path=repo_root)
ik_seconds = time.perf_counter() - started
saved_cfg, result = load_stac_data(ik_path)
assert result.qpos.shape == (len(kp_data), model.nq)
assert result.qvel.shape == (len(kp_data), model.nv)
assert result.kp_names == kp_names
for name in ("qpos", "qvel", "xpos", "xquat", "marker_sites", "offsets"):
    assert np.isfinite(np.asarray(getattr(result, name))).all(), name
np.testing.assert_allclose(result.kp_data, np.asarray(kp_data))
np.testing.assert_allclose(result.offsets, calibration.offsets)

print(f"IK + velocities: {ik_seconds:.1f} s, including any compilation")
print("Saved:", Path(ik_path).name)
print("Solver log:", output_dir / "ik.log")
display(
    pd.DataFrame(
        [
            {
                "stage": "calibration",
                "frames": CALIBRATION_FRAMES,
                "seconds": calibration_seconds,
            },
            {"stage": "IK + velocities", "frames": len(kp_data), "seconds": ik_seconds},
        ]
    ).round(2)
)

## 5. Render the fitted trajectory

The video shows colored observed keypoints and black model markers.
Red connectors show the distance between each pair. They are visual guides;
rendering does not run a physics simulation.

We render the saved IK file, which contains the full selected trajectory.
`RENDER_FPS` controls playback speed. Setting it to the recording rate gives
real-time playback; it does not change the saved velocities.


In [ ]:
camera_names = [model.camera(i).name for i in range(model.ncam)]
assert CAMERA in camera_names, camera_names
print("Available cameras:", ", ".join(camera_names))
video_path = output_dir / "ik.mp4"
_, frames = stac_mjx.viz_stac(
    ik_path,
    n_frames=len(kp_data),
    save_path=video_path,
    camera=CAMERA,
    height=480,
    width=640,
    base_path=repo_root,
    show_marker_error=True,
)
assert len(frames) == len(kp_data)
media.show_video(frames, fps=saved_cfg.model.RENDER_FPS)
print(
    f"Saved {video_path.name}: {len(frames)} frames at {saved_cfg.model.RENDER_FPS} fps"
)

In [ ]:
selected = np.linspace(0, len(frames) - 1, 4, dtype=int)
fig, axes = plt.subplots(1, 4, figsize=(14, 3), layout="constrained")
for ax, frame in zip(axes, selected, strict=True):
    ax.imshow(frames[frame])
    ax.set_title(f"t = {elapsed[frame]:.2f} s")
    ax.axis("off")
fig.suptitle("Fitted rodent, observed keypoints, and marker errors")
fig.savefig(output_dir / "fitted_motion.png", dpi=150)
plt.show()

## 6. Measure the fit

For each frame and marker, we measure Euclidean distance:

$$e_{tk} = 1000\,\lVert \hat y_{tk} - y_{tk} \rVert_2.$$

Positions are in meters, so the factor of 1000 gives millimeters.
The left panel shows mean and maximum marker error over time. The right panel
shows mean error by keypoint. Dashed lines mark IK chunk boundaries.

These distances measure agreement with the recorded keypoints. They do not
measure error against known joint angles. A small distance can coexist with an
implausible pose when observations leave a joint poorly constrained.


In [ ]:
fitted_markers = result.marker_sites.reshape(observed.shape)
errors_mm = np.linalg.norm(fitted_markers - observed, axis=-1) * 1000
per_marker = pd.DataFrame(
    {
        "keypoint": kp_names,
        "mean error (mm)": errors_mm.mean(axis=0),
        "95th percentile (mm)": np.percentile(errors_mm, 95, axis=0),
        "maximum error (mm)": errors_mm.max(axis=0),
    }
).sort_values("mean error (mm)")
per_marker.to_csv(output_dir / "marker_errors.csv", index=False)
display(per_marker.round(3))
print(f"Overall mean marker error: {errors_mm.mean():.3f} mm")

fig, axes = plt.subplots(1, 2, figsize=(13, 6), layout="constrained")
axes[0].plot(elapsed, errors_mm.mean(axis=1), label="Mean", color="#137c8b")
axes[0].plot(
    elapsed, errors_mm.max(axis=1), label="Maximum", color="#dc8b24", alpha=0.8
)
for boundary in range(FRAMES_PER_CHUNK, len(kp_data), FRAMES_PER_CHUNK):
    axes[0].axvline(elapsed[boundary], color="0.6", linestyle="--", linewidth=0.8)
axes[0].set(xlabel="Time from window start (s)", ylabel="Marker error (mm)")
axes[0].legend()
axes[1].barh(per_marker["keypoint"], per_marker["mean error (mm)"], color="#137c8b")
axes[1].set_xlabel("Mean marker error (mm)")
fig.suptitle("Agreement between fitted markers and recorded keypoints")
fig.savefig(output_dir / "marker_errors.png", dpi=150)
plt.show()

### Inspect the largest error

A summary can hide a short failure. We locate the largest marker error, display
that frame, and compare the marker's recorded and fitted coordinates over time.
Look for a persistent offset, a missed movement, or a discontinuity near a chunk
boundary. These patterns suggest different changes to the mapping, calibration,
or solver settings.


In [ ]:
worst_frame, worst_marker = np.unravel_index(np.argmax(errors_mm), errors_mm.shape)
print(
    f"Largest error: {kp_names[worst_marker]}, source frame "
    f"{START_FRAME + worst_frame}, {errors_mm[worst_frame, worst_marker]:.3f} mm"
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].imshow(frames[worst_frame])
axes[0].set_title(f"t = {elapsed[worst_frame]:.2f} s")
axes[0].axis("off")
for axis, color in enumerate(("#137c8b", "#dc8b24", "#6755a4")):
    axes[1].plot(
        elapsed,
        observed[:, worst_marker, axis] * 1000,
        color=color,
        label=f"{'xyz'[axis]} observed",
    )
    axes[1].plot(
        elapsed,
        fitted_markers[:, worst_marker, axis] * 1000,
        color=color,
        linestyle="--",
        label=f"{'xyz'[axis]} fitted",
    )
axes[1].axvline(elapsed[worst_frame], color="0.6", linewidth=0.8)
axes[1].set(
    xlabel="Time from window start (s)",
    ylabel="World coordinate (mm)",
    title=kp_names[worst_marker],
)
axes[1].legend(ncols=2, fontsize=8)
fig.savefig(output_dir / "largest_error.png", dpi=150)
plt.show()

## 7. Inspect joints and saved outputs

The IK file contains more than joint angles. Let $T$ be the frame count,
$K$ the marker count, and $B$ the body count:

| Field | Contents | Shape / units |
|---|---|---|
| `qpos` | Root position, quaternions, and scalar joint coordinates | `(T, nq)`; meters, unit quaternions, radians |
| `qvel` | Translational and angular velocities | `(T, nv)`; m/s and rad/s |
| `xpos` | Body positions in world coordinates | `(T, B, 3)`; meters |
| `xquat` | Body orientations in world coordinates | `(T, B, 4)`; wxyz quaternions |
| `marker_sites` | Fitted marker positions in world coordinates | Inspect the stored shape below; meters |
| `offsets` | Marker positions in their parent bodies | `(K, 3)`; meters |
| `kp_data` | Ordered observations used for the fit | `(T, 3*K)`; meters |

A free joint stores three position coordinates and a four-component quaternion,
but has six velocity components. A ball joint stores four quaternion components
and has three angular velocities. Thus `nq` and `nv` need not match.
MuJoCo quaternions use **w, x, y, z** order.

Names accompany the arrays. `names_qpos` repeats a joint name for each of its
stored position coordinates. Use MuJoCo's `jnt_qposadr` and `jnt_dofadr` to
locate position and velocity columns; do not assume those addresses are equal.

Differentiation magnifies small jumps in fitted poses. Dashed lines below mark
chunk boundaries; inspect nearby velocity spikes alongside the angle and error
curves. Shared context does not enforce exact continuity between chunk solutions.


In [ ]:
display(
    pd.DataFrame(
        [
            {"field": name, "shape": str(np.asarray(getattr(result, name)).shape)}
            for name in (
                "qpos",
                "qvel",
                "xpos",
                "xquat",
                "marker_sites",
                "offsets",
                "kp_data",
            )
        ]
    )
)
joint_labels = ["elbow_L", "elbow_R", "shoulder_L", "shoulder_R"]
joint_ids = [model.joint(name).id for name in joint_labels]
assert all(model.jnt_type[i] == mujoco.mjtJoint.mjJNT_HINGE for i in joint_ids)
qpos_columns = model.jnt_qposadr[joint_ids]
qvel_columns = model.jnt_dofadr[joint_ids]
angles = np.rad2deg(result.qpos[:, qpos_columns])
velocities = np.rad2deg(result.qvel[:, qvel_columns])

fig, axes = plt.subplots(4, 2, figsize=(11, 9), sharex=True, layout="constrained")
for row, (name, joint_id) in enumerate(zip(joint_labels, joint_ids, strict=True)):
    axes[row, 0].plot(elapsed, angles[:, row], color="#137c8b")
    axes[row, 0].set_ylabel(f"{name}\nAngle (°)")
    if model.jnt_limited[joint_id]:
        for limit in np.rad2deg(model.jnt_range[joint_id]):
            axes[row, 0].axhline(limit, color="0.65", linestyle=":", linewidth=0.8)
    axes[row, 1].plot(elapsed, velocities[:, row], color="#6755a4")
    axes[row, 1].set_ylabel("Velocity (°/s)")
for boundary in range(FRAMES_PER_CHUNK, len(kp_data), FRAMES_PER_CHUNK):
    for ax in axes.flat:
        ax.axvline(elapsed[boundary], color="0.8", linestyle="--", linewidth=0.8)
for ax in axes[-1]:
    ax.set_xlabel("Time from window start (s)")
fig.suptitle("Selected hinge joints · dotted lines show model limits")
fig.savefig(output_dir / "joint_motion.png", dpi=150)
plt.show()

### Check the velocity convention

MuJoCo's `mj_differentiatePos` provides an independent check of differentiation
on the fitted poses. We use the same recording timestep and endpoint rule.
This checks the velocity calculation, not whether the fitted movement matches
the animal's true joint motion.

For a free joint, translational velocity is expressed in world coordinates and
angular velocity in the local body frame. Angular velocities have three
components; they are not derivatives of the four quaternion components.


In [ ]:
mujoco_velocity = np.empty_like(result.qvel, dtype=np.float64)
for frame in range(len(result.qpos)):
    previous = min(frame, len(result.qpos) - 2)
    mujoco.mj_differentiatePos(
        model,
        mujoco_velocity[frame],
        1 / saved_cfg.stac.data_fps,
        result.qpos[previous],
        result.qpos[previous + 1],
    )
np.testing.assert_allclose(result.qvel, mujoco_velocity, atol=1e-5, rtol=1e-5)
print(
    "Maximum absolute velocity difference from MuJoCo:",
    f"{np.max(np.abs(result.qvel - mujoco_velocity)):.3g}",
)

## 8. Reuse the workflow

The HDF5 output includes the resolved configuration and the ordered keypoints
used for the fit. `frames.csv` records the source frame indices and timestamps,
which STAC's output does not store separately. Reloading an output does not run
the optimizer.

The next cell checks that reloading preserves the trajectory and lists the
artifacts from this tour. Both calibration and IK files can be passed to
`viz_stac`; the calibration file contains only the calibration frames.


In [ ]:
reloaded_cfg, reloaded = load_stac_data(ik_path)
np.testing.assert_array_equal(reloaded.qpos, result.qpos)
np.testing.assert_array_equal(reloaded.qvel, result.qvel)
assert reloaded.kp_names == kp_names
assert reloaded_cfg.stac.data_fps == cfg.stac.data_fps
assert OmegaConf.to_container(reloaded_cfg.model) == OmegaConf.to_container(cfg.model)
display(
    pd.DataFrame(
        [
            {"file": path.name, "size (KiB)": path.stat().st_size / 1024}
            for path in sorted(output_dir.iterdir())
            if path.is_file()
        ]
    ).round(1)
)

### Fit another segment with the same calibration

The following snippets are optional; **Run All** does not execute them.
Reuse offsets only when the animal, model scale, and marker definitions are
compatible. In this example, the next segment comes from the same recording:

```python
next_cfg = copy.deepcopy(cfg)
next_cfg.stac.skip_calibration = True
next_cfg.stac.ik_path = str(output_dir / "next_segment.h5")
next_keypoints = all_keypoints[stop_frame : stop_frame + 150]
if len(next_keypoints) < 2:
    raise ValueError("Choose a window with at least two remaining frames.")
_, next_ik_path = stac_mjx.run_stac(
    next_cfg, next_keypoints, kp_names, base_path=repo_root
)
```

To run calibration and IK together, leave both `skip_calibration` and
`skip_ik` false and call `run_stac` once.

### Run from the command line

From the repository root, this command reuses the tour's calibration and fits
the **entire bundled recording**. The CLI loads all input frames; the notebook's
frame selection is a Python slice.

```bash
stac-mjx stac=demo model=rodent \
  stac.calibration_path=outputs/rodent_tour/calibration.h5 \
  stac.ik_path=outputs/rodent_tour/full_recording.h5 \
  stac.skip_calibration=true stac.skip_ik=false \
  stac.infer_qvels=true stac.data_fps=30 \
  stac.n_frames_per_clip=50 +stac.q_opt.context_frames=8
```

The leading `+` adds an option that is absent from the YAML preset.

### Adjust the fit or supply your own data

Change one setting at a time, then compare marker errors, joint trajectories,
and the video over the same interval.

| Setting | What it changes |
|---|---|
| `n_calibration_frames`, `N_SAMPLE_FRAMES`, `N_ITERS` | Calibration coverage and number of alternating updates |
| `SITES_TO_REGULARIZE`, `M_REG_COEF` | Which marker offsets resist changes, and how strongly |
| `n_frames_per_clip`, `q_opt.context_frames` | Solve size, memory use, and shared context at chunk boundaries |
| `q_opt.coarse_init_stride` | Spacing of initialization keyframes; zero disables the coarse solve |
| `q_opt.coarse_init_max_frames` | Maximum keyframes per coarse segment when greater than one |
| `q_opt.velocity_smoothness_weight` | Weight of adjacent-pose residuals |
| `q_opt.initial_step_damping` | Initial damping of the pose solver's updates |
| `q_opt.calibration_max_iterations`, `q_opt.ik_max_iterations` | Limits on pose-solver iterations |

For another recording, set `stac.data_path` and `stac.data_fps`, confirm units,
and check the model's keypoint mapping. NWB and supported DANNCE MAT files can
supply names; the HDF5 loader expects its supported `tracks` layout and uses
configured names. These loaders do not accept every file with those extensions.
STAC does not currently use stored confidence scores as fit weights. Resolve missing
coordinates before fitting.

Taking every $s$-th frame changes the input rate to $f/s$. Set `data_fps`
accordingly. If timestamps are irregular, resample onto a uniform time grid
before using this velocity workflow.

For another body, also replace the model preset, joint selections, and camera.
The core steps remain calibration, IK, inspection, and rendering.

**API references:** [pipeline](../stac_mjx/main.py),
[data loading and output](../stac_mjx/io.py),
[configuration](../stac_mjx/config.py), [rendering](../stac_mjx/viz.py).
